# EXP-033 · One Run all on Kaggle

Attach the private P2 inputs and revised runtime ZIP. Select a T4 GPU and enable Internet. Set **AUTHORIZE_RUN = True** and use **Run all**. The worker checks the pretrained model, validates the local resume canary and automatically runs all five folds if those checks pass.

Sealed checkpoints and logs appear in Kaggle Working during execution. A verified archive and SHA-256 receipt are exported at the end, including after a handled failure. Save the output before ending the session. Optional resume uses a previously exported and saved archive.


In [ ]:
AUTHORIZE_RUN = False

# Leave empty to locate the uniquely named ZIPs in the attached private datasets.
INPUT_ZIP = ""
RUNTIME_ZIP = ""
RESUME_ZIP = ""  # optional, path to a previously saved run archive under /kaggle/input
RESUME_SHA256 = ""
MODEL_DIRECTORY = ""  # optional verified weights; otherwise download the pinned model
MAX_SECONDS = 32_400  # worker budget, leaving extra session time for setup and export

SOURCE_COMMIT = "9fdc3eeaf989d394ebb960a441d1d3496ac8f2ae"
RUNTIME_SHA = "055244538f4c1377f6ab9bd3dc0469018016ebf194c77d6894e074d723a11bbb"
INPUT_SHA = "3bfb284878c886c7313875f909171d3d015ee7860790375a7f873e904b106d84"
MANIFEST_SHA = "21f47c3d3c44530b2ea55d12368b46bf1050d665de607094bce591283baf1a82"


In [ ]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import subprocess
import sys
import zipfile


def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def mounted(value):
    path = Path(value).resolve()
    if not value or not path.is_relative_to("/kaggle/input") or not path.is_file():
        raise ValueError(f"Attach and locate the private input under /kaggle/input: {value or '(unset)'}")
    return path


def attached(value, filename):
    if value:
        return mounted(value)
    matches = sorted(Path("/kaggle/input").rglob(filename))
    if len(matches) != 1:
        raise ValueError(f"Expected one attached {filename}. Attach it or set its path explicitly.")
    return mounted(str(matches[0]))


def run():
    if not AUTHORIZE_RUN:
        print("Review the configuration and set AUTHORIZE_RUN = True, then click Run all.")
        return
    if not 0 < MAX_SECONDS <= 36_000:
        raise ValueError("Set a finite worker budget and leave time to export.")
    runtime = attached(RUNTIME_ZIP, "exp033_runtime_oneclick.zip")
    inputs_zip = attached(INPUT_ZIP, "exp033_inputs.zip")
    if sha(runtime) != RUNTIME_SHA or sha(inputs_zip) != INPUT_SHA:
        raise ValueError("Runtime or data ZIP checksum changed")
    attempt = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    local = Path("/kaggle/temp") / f"exp033_{attempt}"
    repo = local / "source"
    output = Path("/kaggle/working") / f"exp033_{attempt}_checkpoints"
    repo.mkdir(parents=True)
    # RUNTIME_SHA pins the exact git archive contents.
    with zipfile.ZipFile(runtime) as bundle:
        names = bundle.namelist()
        for name in names:
            item = PurePosixPath(name)
            if item.is_absolute() or ".." in item.parts or "\\" in name:
                raise ValueError(f"Invalid runtime ZIP member: {name}")
        bundle.extractall(repo)
    helper = repo / "scripts/exp033_artifacts.py"
    if RESUME_ZIP:
        subprocess.run([sys.executable, str(helper), "restore", str(mounted(RESUME_ZIP)),
                        str(output), "--sha", RESUME_SHA256], check=True)
    output.mkdir(exist_ok=True)
    logs = output / f"session_{attempt}.log"
    env = os.environ.copy()
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    env["HF_HOME"] = str(Path("/kaggle/temp") / "exp033_model_cache")
    environment = local / "venv"
    exe = environment / "bin/python"
    artifact = Path("/kaggle/working") / f"exp033_{attempt}.zip"

    def execute(cmd, log):
        process = subprocess.Popen([str(p) for p in cmd], cwd=repo, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="")
            if process.wait() != 0:
                raise RuntimeError(f"Worker exited {process.returncode}, see {logs.name}")
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()

    try:
        with logs.open("w", encoding="utf-8") as log:
            print(f"EXP-033 worker {SOURCE_COMMIT}", file=log)
            gpu = subprocess.run(
                ["nvidia-smi", "-i", "0", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv"],
                capture_output=True, text=True, check=True,
            )
            print(gpu.stdout, file=log, flush=True)
            print(gpu.stdout)
            if "T4" not in gpu.stdout:
                raise ValueError("Select a Kaggle T4 accelerator.")
            bootstrap = local / "uv_pkg"
            execute([sys.executable, "-m", "pip", "install", "--target", bootstrap, "uv==0.8.15"], log)
            env["PYTHONPATH"] = str(bootstrap)
            env["UV_PYTHON_INSTALL_DIR"] = str(Path("/kaggle/temp") / "exp033_python")
            uv = [sys.executable, "-m", "uv"]
            execute(uv + ["python", "install", "3.12.10"], log)
            execute(uv + ["venv", "--python", "3.12.10", environment], log)
            execute(uv + ["pip", "install", "--python", exe, "torch==2.7.1+cu126",
                          "--index-url", "https://download.pytorch.org/whl/cu126"], log)
            execute(uv + ["pip", "install", "--python", exe, "-r", repo / "requirements-exp033.lock.txt"], log)
            execute(uv + ["pip", "check", "--python", exe], log)
            execute(uv + ["pip", "freeze", "--python", exe], log)
            env.pop("PYTHONPATH")
            cmd = [exe, "-u", repo / "scripts/run_exp033.py", "run", "--authorize",
                   "--input-zip", inputs_zip, "--inputs", Path("/kaggle/temp") / f"exp033_inputs_{MANIFEST_SHA[:16]}",
                   "--output", output, "--zip-sha", INPUT_SHA, "--manifest-sha", MANIFEST_SHA,
                   "--cache-dir", Path("/kaggle/temp") / "exp033_frozen", "--max-seconds", MAX_SECONDS]
            if MODEL_DIRECTORY:
                cmd += ["--model-dir", Path(MODEL_DIRECTORY)]
            execute(cmd, log)
            info = json.loads((output / "decision.json").read_text(encoding="utf-8"))
            print("Decision:", info["status"])
            for scope, arms in info.get("metrics", {}).items():
                print(scope, {name: (row["pooled"]["cultural_contrast"], row["pooled"]["log_loss"])
                              for name, row in arms.items()})
    except (Exception, KeyboardInterrupt) as error:
        with logs.open("a", encoding="utf-8") as log:
            print(f"Notebook stopped: {error}", file=log)
        raise
    finally:
        subprocess.run([sys.executable, str(helper), "export", str(output), str(artifact)], check=True)
        print("Export verified:", artifact.name)
        print("Archive SHA-256:", sha(artifact))
        print("Publish or download this archive before stopping the runtime. Local session storage is temporary.")


run()


### Optional resume

Attach a previously exported archive as private input and set its path and SHA-256 in the configuration. The same Run all rechecks T4 feasibility, verifies the original worker and inputs, and resumes from sealed epochs. Fold results already sealed are reused. If Kaggle destroys the session before its output is saved, checkpoints that existed only in that session are unavailable.
